# Title / subtitle dry run — five signal types

Dry run of the alert-card **title** (`ISSUE`) and **subtitle** (`RECOMMENDED_ACTION`)
produced by `rule_based.generate()` for the five action-first signal types:
`OSA_STORE_ACTIONABLE`, `OSA_BIG_DROP`, `MARKDOWN_SKU`, `MARKDOWN_OVERRIDE` and
`OSA_RECURRING_SLOT`. The front-end card renders `ISSUE` bold with
`RECOMMENDED_ACTION` underneath; `EXPLANATION` and `COST_IF_NOT_FIXED` remain the
detail view and are unchanged.

- **Section 1** runs locally on representative enriched signals — no Snowflake needed.
- **Section 2** runs the real pipeline (detection → facts → narrative) live against
  `eastofengland` adapter data with the relevant detectors enabled and per-store caps
  raised, and prints the cards produced right now using the code in this repo.

Nothing is written to Snowflake — this does not read or write `ALERTS_PIPELINE`.

In [ ]:
import sys, os

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

from store_insight.pipeline.stages._4_investigate.rule_based import generate

## 1. Local preview — representative signals

One representative enriched-signal dict per signal type (same shapes as the unit-test
fixtures), printed as TITLE / SUBTITLE / EXPLANATION / COST.

In [ ]:
SCENARIOS = [
    ('OSA_STORE_ACTIONABLE', {
        'signal_type': 'OSA_STORE_ACTIONABLE',
        'store_id': 1,
        'department': 'CHILLED',
        'trigger_types': ['WOW', 'VS_CHAIN'],
        'last_week_osa': 95.0,
        'this_week_osa': 88.0,
        'wow_drift_pp': -7.0,
        'chain_this_week_osa': 93.0,
        'chain_gap_pp': -5.0,
        'this_week_lsv': {'total': 700.0, 'store_controlled': 500.0, 'upstream': 180.0, 'unknown': 20.0},
        'facts': {
            'item_lsv_changes': [{
                'item_code': '10', 'item_name': 'Milk', 'eans': ['501'], 'upcs': [],
                'lsv_change': {'store_controlled': 25.0},
            }],
            'worst_day': {'day': 'Friday'},
            'cost_projection': {'projection_weeks': 4, 'store_controlled_cost_of_inaction': 2000.0},
        },
    }),
    ('OSA_BIG_DROP', {
        'signal_type': 'OSA_BIG_DROP',
        'store_id': 1,
        'department': 'SPIRITS',
        'trigger_types': ['BIG_DROP'],
        'last_week_osa': 95.0,
        'this_week_osa': 88.0,
        'wow_drift_pp': -7.0,
        'latest_day': 'Sunday',
        'prior_day': 'Saturday',
        'this_week_lsv': {'total': 700.0, 'store_controlled': 500.0, 'upstream': 180.0, 'unknown': 20.0},
        'facts': {
            'worst_day_period': {'day': 'Saturday', 'period_id': 3, 'period_label': 'Evening (6pm-Close)'},
            'ppg_lsv_changes': [{'ppg': 'Vodka', 'latest_week_lsv': {'store_controlled': 120.0}}],
            'item_lsv_changes': [{
                'item_code': '10', 'item_name': 'Smirnoff', 'eans': ['501'], 'upcs': [],
                'lsv_change': {'store_controlled': 25.0},
            }],
            'cost_projection': {'projection_weeks': 4, 'store_controlled_cost_of_inaction': 2000.0},
        },
    }),
    ('MARKDOWN_SKU', {
        'signal_type': 'MARKDOWN_SKU',
        'store_id': 1,
        'site_name': 'Store 1',
        'item_id': 42,
        'item_description': 'Milk 2L',
        'department': 'CHILLED',
        'trigger_types': ['REPEAT_REQUEST'],
        'markdown_request_count': 6,
        'override_count': 2,
        'lookback_weeks': 4,
        'avg_markdown_pct': 30.0,
        'facts': {'sku_markdown_history': []},
    }),
    ('MARKDOWN_OVERRIDE', {
        'signal_type': 'MARKDOWN_OVERRIDE',
        'store_id': 1,
        'site_name': 'Store 1',
        'trigger_types': ['VS_CHAIN'],
        'this_week_override_rate': 25.0,
        'last_week_override_rate': 10.0,
        'wow_change_pp': 15.0,
        'chain_override_rate': 10.0,
        'above_chain_pp': 15.0,
        'this_week_total_markdowns': 40,
        'this_week_overrides': 10,
        'facts': {'override_dept_breakdown': [
            {'dept': 'BAKERY', 'overrides': 5, 'override_rate_pct': 50.0},
        ]},
    }),
    ('OSA_RECURRING_SLOT', {
        'signal_type': 'OSA_RECURRING_SLOT',
        'store_id': 42,
        'department': 'PRODUCE',
        'dow': 2,
        'dow_name': 'Monday',
        'period_id': 1,
        'period_label': 'Morning (6am-12pm)',
        'trigger_types': ['RECURRING_SLOT'],
        'repeat_item_count': 2,
        'repeat_item_details': [
            {'item_code': 'APL001', 'item_description': 'Gala Apples', 'repeat_weeks': 4, 'total_lsv': 350.0, 'avg_weekly_lsv': 87.5},
            {'item_code': 'BAN002', 'item_description': 'Bananas', 'repeat_weeks': 3, 'total_lsv': 210.0, 'avg_weekly_lsv': 70.0},
        ],
        'total_lsv': 560.0,
        'avg_weekly_lsv': 157.5,
        'facts': {
            'slot_history': [
                {'year': 2025, 'week': 1, 'sales': 80.0, 'lsv_vol': 20.0, 'lsv': 100.0, 'osa_pct': 80.0},
                {'year': 2025, 'week': 2, 'sales': 75.0, 'lsv_vol': 25.0, 'lsv': 125.0, 'osa_pct': 75.0},
                {'year': 2025, 'week': 3, 'sales': 70.0, 'lsv_vol': 30.0, 'lsv': 150.0, 'osa_pct': 70.0},
            ],
            'recurring_slot_share_pct': 45.0,
            'slot_share_of_dept_total_pct': 12.5,
            'slot_trend': 'increasing',
        },
    }),
]

In [ ]:
for label, signal in SCENARIOS:
    out = generate(signal)
    print('=' * 78)
    print(label)
    print('=' * 78)
    print(f'TITLE      : {out["issue"]}')
    print(f'SUBTITLE   : {out["recommended_action"]}')
    print(f'EXPLANATION: {out["explanation"]}')
    print(f'COST       : {out["cost_if_not_fixed"]}')
    print()

## 2. Live run — real pipeline against eastofengland

Runs detection → facts → narrative generation live against current adapter data with
the five relevant detector groups enabled, and per-store caps raised
(`top_n_per_store=50`, `max_per_type_per_store=50`) so no type is squeezed out.
This does **not** read the stored `ALERTS_PIPELINE` table and writes nothing.

Set `ANALYSIS_DATE` to `None` for the latest OSA date, or a specific `'YYYY-MM-DD'`.
`STORE_IDS` can be a list of store ids to keep the run fast.

In [ ]:
CLIENT = 'eastofengland'
ANALYSIS_DATE = None
STORE_IDS = [451, 497, 463, 416, 479, 637, 426, 440, 239, 445, 211, 634, 209, 466, 492, 633, 444, 499, 507, 505, 448, 471, 224, 418]  

DETECTORS = [
    'osa_store_actionable.wow',
    'osa_store_actionable.vs_chain',
    'osa_big_drop.big_drop',
    'osa_recurring_slot.recurring_slot',
    'markdown_override.vs_chain',
    'markdown_override.wow',
    'markdown_sku.repeat_request',
    'markdown_sku.excessive_override',
]

import logging, warnings

logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session
from store_insight.pipeline.pipeline.capabilities import discover_capabilities

cfg = load_config(CLIENT)
capabilities = discover_capabilities(cfg.client_name)

cfg = cfg.model_copy(update={
    'detectors': cfg.detectors.model_copy(update={'enabled': DETECTORS}),
    'output': cfg.output.model_copy(update={
        'top_n_per_store': 50,
        'max_per_type_per_store': 50,
    }),
})

print(f'Config loaded: client={cfg.client_name}')
print(f'Enabled detectors: {cfg.detectors.enabled}')
print(f'top_n_per_store={cfg.output.top_n_per_store}, '
      f'max_per_type_per_store={cfg.output.max_per_type_per_store}')
session = create_session(cfg)

In [ ]:
from snowflake.snowpark import functions as F

if ANALYSIS_DATE:
    analysis_date = ANALYSIS_DATE
else:
    latest = (
        session.table(cfg.tables.osa)
        .select(F.max('DATE').alias('LATEST_DATE'))
        .collect()[0]['LATEST_DATE']
    )
    if latest is None:
        raise RuntimeError(f'No OSA data found in {cfg.tables.osa}')
    analysis_date = str(latest)

print(f'Analysis date: {analysis_date}')

In [ ]:
from store_insight.pipeline.pipeline.pipeline import derive_windows
from store_insight.pipeline.stages._0_quality.quality_gate import run_quality_gate
from store_insight.pipeline.pipeline.run_batch import load_feedback, load_osa, load_wmd

windows = derive_windows(analysis_date, cfg)
for name, value in windows.items():
    print(f'{name:18} {value}')

gate_plan = run_quality_gate(session, cfg, analysis_date=analysis_date)

osa = load_osa(session, windows['lookback_start'], windows['this_week_end'], cfg=cfg)
feedback = load_feedback(
    session, windows['last_week_start'], windows['this_week_end'],
    cfg=cfg, capabilities=capabilities,
)
wmd = load_wmd(
    session, windows['wmd_lookback'], windows['this_week_end'],
    cfg=cfg, capabilities=capabilities,
)
print(f'OSA rows: {osa.count():,}')

In [ ]:
import os
import uuid
from store_insight.pipeline.pipeline.pipeline import run_pipeline

cards, raw_signals, enriched_signals = run_pipeline(
    osa,
    last_week_start=windows['last_week_start'],
    last_week_end=windows['last_week_end'],
    this_week_start=windows['this_week_start'],
    this_week_end=windows['this_week_end'],
    lookback_start=windows['lookback_start'],
    store_id=STORE_IDS,
    session=session,
    feedback=feedback,
    markdown_log=wmd['markdown_log'],
    markdown_sales=wmd['markdown_sales'],
    wastage=wmd['wastage'],
    item_hierarchy=wmd['item_hierarchy'],
    store_hierarchy=wmd['store_hierarchy'],
    cfg=cfg,
    return_intermediates=True,
    run_id=str(uuid.uuid4()),
    pipeline_version=os.environ.get('PIPELINE_VERSION'),
    gate_plan=gate_plan,
)

print(f'Raw signals: {len(raw_signals):,}')
print(f'Cards: {len(cards):,}')

In [ ]:
from collections import Counter, defaultdict

TARGET_TYPES = [
    'OSA_STORE_ACTIONABLE',
    'OSA_BIG_DROP',
    'MARKDOWN_SKU',
    'MARKDOWN_OVERRIDE',
    'OSA_RECURRING_SLOT',
]

by_type = defaultdict(list)
for card in cards:
    by_type[(card.get('signal_json') or {}).get('signal_type')].append(card)

print('Card counts per signal type:')
for stype, count in Counter(
    (c.get('signal_json') or {}).get('signal_type') for c in cards
).most_common():
    print(f'  {stype}: {count}')
print()

for stype in TARGET_TYPES:
    group = sorted(by_type.get(stype, []), key=lambda c: c.get('sort_value') or 0, reverse=True)
    print('=' * 78)
    print(f'{stype} — {len(group)} card(s)')
    print('=' * 78)
    if not group:
        print(f'No {stype} cards produced for this analysis date/store set.\n')
        continue
    for card in group[:3]:
        sig = card.get('signal_json') or {}
        print(f"store {card['store_id']} | {card.get('section')} | "
              f"sort_value={card.get('sort_value')}")
        print(f'TITLE      : {card["issue"]}')
        print(f'SUBTITLE   : {card["recommended_action"]}')
        print(f'EXPLANATION: {card["explanation"]}')
        print(f'COST       : {card["cost_if_not_fixed"]}')
        print('-' * 78)
    print()

In [ ]:
session.close()
print('Snowflake session closed.')